# Фінальний проєкт (Тема 10): Інтелектуальне збагачення Excel-даних
**Мета:** Створення універсальної автономної системи інтелектуального збагачення табличних даних Excel на базі LangGraph / LangChain та гібридного веб-пошуку.

---

### Архітектура та функціонал системи:
1. **Аналізатор структури та планувальник (Analyzer Agent)**:
   - Аналізує структуру вхідного Excel-файлу.
   - Розбирає текстовий опис завдання користувача та формує план збагачення (`TaskAnalysisPlan`).
   - Визначає цільову колонку, контекстні колонки, тип даних та оптимізований шаблон пошукового запиту.
2. **Hybrid Web Search Engine**:
   - Інтелектуальний веб-пошук DuckDuckGo (`ddgs`) для отримання актуальних фактів.
   - Пошук через Wikipedia API для енциклопедичних та географічних відомостей.
   - Калькулятор (`geopy.Nominatim` + Geodesic Distance) для точності обчислення відстаней між містами/координатами.
3. **Extraction & Validation Agent**:
   - LLM зі структурованим виводом (`ExtractedValue`) витягує чисте цільове значення з результатів пошуку.
   - Автоматичне приведення типів (`float` / `int` для числових колонок) та багаторівнева обробка помилок.
4. **Паралельний оркестратор: ThreadPoolExecutor + LangGraph Workflow**:
   - Паралельна багатопотокова обробка рядків із масштабованістю до 1000+ записів.
   - Збереження структури: всі оригінальні колонки та значення залишаються незмінними.
5. **Публічний інтерфейс: `process_excel`**:
   - виклик для виконання будь-яких завдань збагачення: `process_excel(file_path, task_description)`.

In [22]:
# !pip install pandas openpyxl langgraph langchain-openai langchain-core python-dotenv pydantic ddgs geopy tqdm

import os
import json
import re
import time
import urllib.parse
import urllib.request
from typing import List, Optional, Dict, Any, Literal, Union, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

from ddgs import DDGS
from geopy.geocoders import Nominatim
from geopy.distance import geodesic


openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_base_url or not openrouter_api_key:
    raise ValueError("OPENROUTER_BASE_URL або OPENROUTER_API_KEY не знайдено у файлі .env")

# Ініціалізація моделі LLM через OpenRouter
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    temperature=0.0
)

print(f"Ініціалізація LLM ({llm.model_name}) успішна.")
print(f"Версія pandas: {pd.__version__}")

Ініціалізація LLM (openai/gpt-4o-mini) успішна.
Версія pandas: 3.1.1


## 1. Pydantic Schema & LangGraph State

Визначаємо структури для автоматичного аналізу завдання та отримання цільових значень за допомогою  `with_structured_output`:
* `TaskAnalysisPlan` — структурований план збагачення: цільова колонка, контекстні колонки, тип даних, одиниці вимірювання та оптимізований шаблон пошукового запиту.
* `ExtractedValue` — результат екстракції значення для конкретного рядка із прапорцем наявності даних та підсумком джерела.
* `EnrichmentState` — стан графа LangGraph для відстеження процесу обробки таблиці.

In [23]:
class TaskAnalysisPlan(BaseModel):
    """Схема структурованого плану збагачення Excel-таблиці"""
    target_column: str = Field(
        description="Назва колонки для заповнення або додавання у DataFrame (наприклад, 'distance', 'height')"
    )
    input_columns: List[str] = Field(
        description="Список назв існуючих колонок, які надають контекст для пошуку"
    )
    data_type: Literal["numeric", "text", "category"] = Field(
        description="Очікуваний тип даних цільової колонки ('numeric', 'text' або 'category')"
    )
    unit: Optional[str] = Field(
        default=None,
        description="Одиниця вимірювання, якщо застосовно (наприклад, 'km', 'meters', 'USD' тощо)"
    )
    query_template: str = Field(
        description="Лаконічний шаблон пошукового запиту з плейсхолдерами назв колонок, наприклад: '{Mountain} mountain {Country} height meters' або 'distance from {Capital_From} {Country_From} to {Capital_To} {Country_To} km'"
    )
    extraction_instructions: str = Field(
        description="Чіткі інструкції щодо вилучення цільового значення з результатів пошуку"
    )


class ExtractedValue(BaseModel):
    """Схема вилученого значення для окремого рядка"""
    value: Optional[Union[float, int, str]] = Field(
        default=None,
        description="Очищене цільове значення без зайвого тексту чи одиниць вимірювання"
    )
    found: bool = Field(
        description="True, якщо достовірна інформація знайдена, False інакше"
    )
    source_summary: Optional[str] = Field(
        default=None,
        description="Коротке резюме джерела або обґрунтування значення"
    )


class EnrichmentState(dict):
    """Словник стану для графа LangGraph"""
    file_path: str
    task_description: str
    max_workers: int
    output_path: Optional[str]
    df: Optional[pd.DataFrame]
    plan: Optional[TaskAnalysisPlan]
    status_message: str
    enriched_df: Optional[pd.DataFrame]


structured_analyzer = llm.with_structured_output(TaskAnalysisPlan)
structured_extractor = llm.with_structured_output(ExtractedValue)

print("Pydantic-моделі та структуровані екстрактори сконфігуровано.")

Pydantic-моделі та структуровані екстрактори сконфігуровано.


## 2. Search & Research Engine

Гібридний підхід:
1. **DuckDuckGo Search (`DDGS`)**: швидкий пошук текстових фактів, висот, числових характеристик.
2. **Wikipedia API**: пошук енциклопедичних описів географічних об'єктів без обмежень на кількість запитів.
3. **Геодезичний калькулятор (`Nominatim` + `geodesic`)**: спеціалізований інструмент для обчислення точної прямої ортодромічної відстані (Great-Circle Distance) між географічними пунктами.
4. **In-Memory Cache**: кешування координат та пошукових запитів для оптимізації швидкості та виключення дублювання запитів.

In [24]:
# геокодер та кеші
geolocator = Nominatim(user_agent="goit_excel_enrichment_system")
_geo_cache: Dict[str, Optional[Tuple[float, float]]] = {}
_search_cache: Dict[str, str] = {}


def get_geo_coordinates(location_name: str) -> Optional[Tuple[float, float]]:
    """Отримання географічних координат (широта, довгота) з кешуванням"""
    cleaned_name = location_name.strip()
    if cleaned_name in _geo_cache:
        return _geo_cache[cleaned_name]

    try:
        loc = geolocator.geocode(cleaned_name, timeout=5)
        if loc:
            _geo_cache[cleaned_name] = (loc.latitude, loc.longitude)
            return _geo_cache[cleaned_name]
    except Exception as e:
        # Логування та безпечний fallback
        pass

    _geo_cache[cleaned_name] = None
    return None


def compute_geodesic_distance(from_location: str, to_location: str) -> Optional[float]:
    """Обчислення прямої геодезичної відстані між двома пунктами у кілометрах"""
    coords_from = get_geo_coordinates(from_location)
    coords_to = get_geo_coordinates(to_location)

    if coords_from and coords_to:
        distance_km = geodesic(coords_from, coords_to).km
        return round(distance_km, 1)
    return None


def search_duckduckgo(query: str, max_results: int = 3) -> str:
    """Виконання веб-пошуку через DuckDuckGo із кешуванням та очищенням запиту"""
    clean_query = re.sub(r"[^\w\s]", " ", query).strip()
    clean_query = re.sub(r"\s+", " ", clean_query)

    if clean_query in _search_cache:
        return _search_cache[clean_query]

    try:
        results = list(DDGS().text(clean_query, max_results=max_results))
        if results:
            snippets = []
            for r in results:
                title = r.get("title", "")
                body = r.get("body", "")
                if body:
                    snippets.append("Title: " + str(title) + chr(10) + "Snippet: " + str(body))
            combined = (chr(10) + chr(10)).join(snippets)
            _search_cache[clean_query] = combined
            return combined
    except Exception as e:
        pass

    return ""


def search_wikipedia_api(query: str) -> str:
    """Швидкий пошук фактів через офіційне API Вікіпедії"""
    try:
        clean_q = query.strip()
        encoded = urllib.parse.quote(clean_q)
        url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={encoded}&utf8=&format=json"
        req = urllib.request.Request(url, headers={"User-Agent": "ExcelDataEnricher/1.0"})
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            search_items = data.get("query", {}).get("search", [])
            if search_items:
                snippets = []
                for item in search_items[:2]:
                    clean_snippet = re.sub(r"<[^>]+>", "", item.get("snippet", ""))
                    snippets.append("Wikipedia [" + str(item.get("title")) + "]: " + str(clean_snippet))
                return chr(10).join(snippets)
    except Exception:
        pass
    return ""


def gather_research_context(row_dict: Dict[str, Any], plan: TaskAnalysisPlan) -> str:
    """Уніфікований збір контексту: пряма геодезична перевірка + веб-пошук + Вікіпедія"""
    contexts = []

    # Спеціалізована перевірка для завдань на обчислення відстаней між містами
    is_distance_task = (
        ("distance" in plan.target_column.lower() or "відстан" in plan.extraction_instructions.lower()) and
        any("capital" in col.lower() or "city" in col.lower() or "from" in col.lower() for col in plan.input_columns)
    )

    if is_distance_task:
        from_locs = [str(row_dict[k]) for k in plan.input_columns if "from" in k.lower() and pd.notna(row_dict.get(k))]
        to_locs = [str(row_dict[k]) for k in plan.input_columns if "to" in k.lower() and pd.notna(row_dict.get(k))]

        from_str = ", ".join(from_locs) if from_locs else ""
        to_str = ", ".join(to_locs) if to_locs else ""

        if from_str and to_str:
            geo_dist = compute_geodesic_distance(from_str, to_str)
            if geo_dist is not None:
                contexts.append(f"Calculated Geodesic Great-Circle Distance between '{from_str}' and '{to_str}': {geo_dist} km.")

    # Формування пошукового запиту за шаблоном плану
    try:
        query = plan.query_template.format(**{k: str(v) if pd.notna(v) else "" for k, v in row_dict.items()})
    except Exception:
        query = " ".join(f"{row_dict.get(col, '')}" for col in plan.input_columns if pd.notna(row_dict.get(col)))

    # 1. Пошук у DuckDuckGo
    ddg_res = search_duckduckgo(query, max_results=3)
    if ddg_res:
        contexts.append("Web Search Results:" + chr(10) + ddg_res)

    # 2. Пошук у Wikipedia, якщо інформації замало
    if len(contexts) < 2:
        wiki_res = search_wikipedia_api(query)
        if wiki_res:
            contexts.append("Wikipedia Results:" + chr(10) + wiki_res)

    return (chr(10) + chr(10)).join(contexts) if contexts else "No external search results found."


## 3. Модуль інтелектуальної екстракції та валідації рядка (Row Processing & Extraction Agent)

Кожен рядок таблиці обробляється незалежним агентом екстракції:
1. Збирається актуальний контекст (веб-пошук + геодані).
2. LLM через `structured_extractor` витягує точне значення згідно з інструкціями плану.
3. Виконується валідація типу даних (числові значення приводяться до `float`/`int`, відкидаються випадкові текстові примітки).
4. У разі відсутності даних або збою зовнішніх API повертається `np.nan` з інформативним повідомленням.

In [25]:
def process_single_row(row_dict: Dict[str, Any], plan: TaskAnalysisPlan) -> Any:
    """Обробка та збагачення окремого рядка таблиці з валідацією та обробкою помилок"""
    try:
        # 1. Збір пошукового та фактичного контексту
        context_data = gather_research_context(row_dict, plan)

        # 2. Формування вхідних значень для промпту
        row_context_json = json.dumps(
            {k: row_dict[k] for k in plan.input_columns if k in row_dict and pd.notna(row_dict[k])},
            ensure_ascii=False
        )

        prompt = f"""Ти — високоточний агент інтелектуального вилучення даних.
Твоє завдання: вилучити точне цільове значення для запису на основі наданого контексту та пошукових даних.

Вхідний запис: {row_context_json}
Цільова колонка: {plan.target_column}
Очікуваний тип: {plan.data_type}
Одиниця вимірювання: {plan.unit or 'без одиниць'}
Спеціальні інструкції: {plan.extraction_instructions}

Знайдений контекст та дані з інтернету:
{context_data}

Вимоги:
- Надай лише чисте цільове значення.
- Якщо очікується числове значення, вкажи виключно число (наприклад, 8848.86 або 2032). Не пиши слів 'km', 'meters', 'approx' тощо у полі значення.
- Якщо точні дані відсутні у пошуку, але ти маєш достовірні енциклопедичні знання, використай їх із зазначенням джерела.
- Якщо знайти достовірне значення неможливо, встанови found=False."""

        res: ExtractedValue = structured_extractor.invoke(prompt)

        if res.found and res.value is not None:
            raw_val = res.value

            # Числова валідація та очищення
            if plan.data_type == "numeric":
                if isinstance(raw_val, (int, float)):
                    return round(float(raw_val), 2) if isinstance(raw_val, float) and "." in str(raw_val) else int(raw_val)
                elif isinstance(raw_val, str):
                    # Пошук першого валідного числа
                    clean_str = raw_val.replace(",", "").replace(" ", "").strip()
                    nums = re.findall(r"[-+]?\d*\.\d+|\d+", clean_str)
                    if nums:
                        parsed = float(nums[0])
                        return int(parsed) if parsed.is_integer() else round(parsed, 2)
                return np.nan

            return str(raw_val).strip()

    except Exception as e:
        print(f"⚠️ [Помилка обробки рядка {row_dict.get(plan.input_columns[0] if plan.input_columns else '', '')}]: {e}")

    return np.nan


## 4. Пайплайн LangGraph (StateGraph Nodes & Workflow)

Побудова повноцінного агентного графа LangGraph:
* **Вузол 1: `analyze_structure_node`** — аналізує структуру датасету та формує план дій.
* **Вузол 2: `parallel_enrich_node`** — виконує багатопотокове паралельне збагачення за допомогою `ThreadPoolExecutor`.
* **Вузол 3: `save_and_verify_node`** — зберігає збагачені дані в Excel-файл, зберігаючи всю оригінальну структуру даних без змін.

In [26]:
def analyze_structure_node(state: EnrichmentState) -> dict:
    """Вузол 1: Аналіз структури Excel-файлу та генерація плану збагачення"""
    file_path = state["file_path"]
    task_description = state["task_description"]

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл '{file_path}' не знайдено.")

    df = pd.read_excel(file_path)
    sample_preview = df.head(3).to_string()
    columns_list = ", ".join(df.columns)

    analysis_prompt = f"""Проаналізуй структуру вхідного набору даних та завдання користувача для створення структурованого плану збагачення.

Наявні колонки: {columns_list}
Зразок перших 3 рядків:
{sample_preview}

Завдання користувача: "{task_description}"

Вимоги до плану:
1. Визнач цільову колонку (target_column) для заповнення або створення.
2. Визнач список вхідних колонок (input_columns), що дають контекст для пошуку.
3. Вкажи тип даних (data_type: 'numeric', 'text' або 'category') та одиницю вимірювання (unit: 'km', 'meters' тощо).
4. Склади короткий, чіткий шаблон пошукового запиту (query_template), використовуючи фігурні дужки {{Column_Name}}.
5. Надай лаконічні інструкції щодо вилучення значення (extraction_instructions)."""

    plan: TaskAnalysisPlan = structured_analyzer.invoke(analysis_prompt)

    print()
    print(f"[Вузол 1: Аналіз] Сформовано план збагачення для '{file_path}':")
    print(f"  • Цільова колонка: {plan.target_column} (Тип: {plan.data_type}, Одиниця: {plan.unit or 'немає'})")
    print(f"  • Вхідні колонки: {plan.input_columns}")
    print(f"  • Шаблон запиту: {plan.query_template}")

    return {
        "df": df,
        "plan": plan,
        "status_message": f"Структуру проаналізовано. Цільова колонка: {plan.target_column}"
    }


def parallel_enrich_node(state: EnrichmentState) -> dict:
    """Вузол 2: Паралельне багатопотокове збагачення рядків таблиці"""
    df = state["df"].copy()
    plan: TaskAnalysisPlan = state["plan"]
    max_workers = state.get("max_workers", 5)

    target_col = plan.target_column
    if target_col not in df.columns:
        df[target_col] = np.nan

    rows = df.to_dict(orient="records")
    total_rows = len(rows)
    results = [None] * total_rows

    start_time = time.time()
    print()
    print(f"[Вузол 2: Збагачення] Обробка {total_rows} записів у {max_workers} потоків...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_single_row, row, plan): idx
            for idx, row in enumerate(rows)
        }

        completed_count = 0
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                val = future.result()
                results[idx] = val
                completed_count += 1
                sample_key = str(rows[idx].get(plan.input_columns[0] if plan.input_columns else '', ''))
                print(f"  [{completed_count}/{total_rows}] Рядок #{idx+1} ({sample_key}): {val}")
            except Exception as e:
                print(f"  NB! Помилка у рядку #{idx+1}: {e}")
                results[idx] = np.nan

    elapsed = time.time() - start_time
    df[target_col] = results

    successful_fill = df[target_col].notna().sum()
    print()
    print(f"[Вузол 2: Завершено] Збагачено {successful_fill}/{total_rows} рядків за {elapsed:.2f} сек.")

    return {
        "enriched_df": df,
        "status_message": f"Збагачення завершено: {successful_fill}/{total_rows} заповнено."
    }


def save_and_verify_node(state: EnrichmentState) -> dict:
    """Вузол 3: Збереження збагаченого Excel-файлу та фінальна валідація"""
    enriched_df: pd.DataFrame = state["enriched_df"]
    file_path = state["file_path"]
    custom_output = state.get("output_path")

    if custom_output:
        final_output_path = custom_output
    else:
        base_name, ext = os.path.splitext(file_path)
        final_output_path = f"{base_name}_enriched{ext}"

    # Збереження у формат Excel
    enriched_df.to_excel(final_output_path, index=False)

    print()
    print(f"[Вузол 3: Збереження] Збагачений файл успішно збережено: '{final_output_path}'")

    return {
        "output_path": final_output_path,
        "status_message": f"Файл успішно збережено за адресою {final_output_path}"
    }


def build_enrichment_graph():
    """Створення та компіляція LangGraph графа збагачення даних"""
    workflow = StateGraph(EnrichmentState)

    workflow.add_node("analyze_structure", analyze_structure_node)
    workflow.add_node("parallel_enrich", parallel_enrich_node)
    workflow.add_node("save_and_verify", save_and_verify_node)

    workflow.add_edge(START, "analyze_structure")
    workflow.add_edge("analyze_structure", "parallel_enrich")
    workflow.add_edge("parallel_enrich", "save_and_verify")
    workflow.add_edge("save_and_verify", END)

    return workflow.compile()

enrichment_app = build_enrichment_graph()
print("Граф LangGraph успішно скомпільовано.")

Граф LangGraph успішно скомпільовано.


## 5. Головний публічний інтерфейс системи (`process_excel`)

Реалізація згідно з вимогами завдання:
```python
process_excel(
    file_path="capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance"
)
```
Функція підтримує:
* Автоматичне визначення шляху до файлів (у корені або у папці `data/`).
* Багатопотокове збагачення з налаштовуваним пулом воркерів.
* Збереження структури та формату `.xlsx`.
* Інформативний вивід прогресу та результуючої таблиці.

In [27]:
def process_excel(
    file_path: str,
    task_description: str,
    max_workers: int = 5,
    output_path: Optional[str] = None
) -> str:
    """
    Головна функція інтелектуального збагачення даних у файлі Excel.

    Параметри:
    ----------
    file_path : str
        Шлях до вхідного Excel-файлу (.xlsx).
    task_description : str
        Текстовий опис завдання природною мовою.
    max_workers : int, optional
        Кількість паралельних потоків для веб-пошуку та обробки (за замовчуванням 5).
    output_path : str, optional
        Шлях для збереження результуючого файлу (якщо None, створюється '<ім'я>_enriched.xlsx').

    Повертає:
    -------
    str : Шлях до збереженого збагаченого Excel-файлу.
    """
    # Автоматична перевірка наявності файлу в поточній директорії або папці data/
    resolved_path = file_path
    if not os.path.exists(resolved_path):
        data_subpath = os.path.join("data", file_path)
        if os.path.exists(data_subpath):
            resolved_path = data_subpath
        else:
            raise FileNotFoundError(f"Не вдалося знайти файл за шляхами '{file_path}' або '{data_subpath}'")

    print("=" * 70)
    print(f" Запуск інтелектуального збагачення: '{resolved_path}'")
    print(f" Завдання: \"{task_description}\"")
    print("=" * 70)

    initial_state = {
        "file_path": resolved_path,
        "task_description": task_description,
        "max_workers": max_workers,
        "output_path": output_path,
        "df": None,
        "plan": None,
        "status_message": "Ініціалізація...",
        "enriched_df": None
    }

    # Виконання графа LangGraph
    final_state = enrichment_app.invoke(initial_state)

    saved_file = final_state.get("output_path")
    enriched_df = final_state.get("enriched_df")

    print()
    print("=" * 70)
    print(f"✅ Збагачення завершено! Файл збережено: {saved_file}")
    print("=" * 70)

    if enriched_df is not None:
        print()
        print("📊 Результат виконання (перші 10 рядків):")
        display(enriched_df.head(10)) if "display" in globals() else print(enriched_df.head(10).to_string())

    return saved_file

## 6. Демонстрація та верифікація на тестових наборах

### Тестовий набір 1: Столиці (`capitals.xlsx`)
**Завдання:** «знайди пряму відстань між столицями в км для колонки distance»

Тестуємо виклик системи з автоматичним заповненням колонки `distance` геодезичними ортодромічними відстанями між європейськими столицями.

In [28]:
# Виклик системи для Набору 1: Столиці
capitals_enriched_file = process_excel(
    file_path="data/capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    max_workers=5
)

 Запуск інтелектуального збагачення: 'data/capitals.xlsx'
 Завдання: "знайди пряму відстань між столицями в км для колонки distance"

[Вузол 1: Аналіз] Сформовано план збагачення для 'data/capitals.xlsx':
  • Цільова колонка: distance (Тип: numeric, Одиниця: km)
  • Вхідні колонки: ['Capital_From', 'Country_From', 'Capital_To', 'Country_To']
  • Шаблон запиту: find distance from {Capital_From} {Country_From} to {Capital_To} {Country_To}

[Вузол 2: Збагачення] Обробка 10 записів у 5 потоків...
  [1/10] Рядок #3 (Paris): 878
  [2/10] Рядок #1 (Kyiv): 2031.9
  [3/10] Рядок #4 (Berlin): 523.9
  [4/10] Рядок #2 (London): 1435.6
  [5/10] Рядок #8 (Vienna): 214.7
  [6/10] Рядок #5 (Warsaw): 687.3
  [7/10] Рядок #7 (Madrid): 503.9
  [8/10] Рядок #6 (Rome): 1366.4
  [9/10] Рядок #9 (Stockholm): 418.8
  [10/10] Рядок #10 (Athens): 561

[Вузол 2: Завершено] Збагачено 10/10 рядків за 9.32 сек.

[Вузол 3: Збереження] Збагачений файл успішно збережено: 'data/capitals_enriched.xlsx'

✅ Збагачення зав

### Тестовий набір 2: Гори (`mountains.xlsx`)
**Завдання:** «додай висоту гір у метрах до колонки height»

Тестуємо виклик тієї ж універсальної системи без будь-яких змін коду для збагачення гір висотами у метрах.

In [29]:
# Виклик системи для Набору 2: Гори
mountains_enriched_file = process_excel(
    file_path="data/mountains.xlsx",
    task_description="додай висоту гір у метрах до колонки height",
    max_workers=5
)

 Запуск інтелектуального збагачення: 'data/mountains.xlsx'
 Завдання: "додай висоту гір у метрах до колонки height"

[Вузол 1: Аналіз] Сформовано план збагачення для 'data/mountains.xlsx':
  • Цільова колонка: height (Тип: numeric, Одиниця: meters)
  • Вхідні колонки: ['Mountain', 'Country']
  • Шаблон запиту: height of {Mountain} in {Country} meters

[Вузол 2: Збагачення] Обробка 10 записів у 5 потоків...
  [1/10] Рядок #5 (Makalu): 8485
  [2/10] Рядок #3 (Kangchenjunga): 8586
  [3/10] Рядок #2 (K2): 8611
  [4/10] Рядок #4 (Lhotse): 8516
  [5/10] Рядок #1 (Mount Everest): 8848.86
  [6/10] Рядок #6 (Cho Oyu): 8188
  [7/10] Рядок #7 (Dhaulagiri): 8167
  [8/10] Рядок #8 (Manaslu): 8163
  [9/10] Рядок #9 (Nanga Parbat): 8126
  [10/10] Рядок #10 (Annapurna): 8091

[Вузол 2: Завершено] Збагачено 10/10 рядків за 7.76 сек.

[Вузол 3: Збереження] Збагачений файл успішно збережено: 'data/mountains_enriched.xlsx'

✅ Збагачення завершено! Файл збережено: data/mountains_enriched.xlsx

📊 Результат 

## 7. Фінальна верифікація

In [30]:
# Перевірка згенерованих файлів та виведення фінальних звітів
print("=== ФІНАЛЬНА ПЕРЕВІРКА РЕЗУЛЬТАТІВ ===")

# Перевіряємо та зчитуємо збережені файли
df_cap_res = pd.read_excel(capitals_enriched_file)
df_mtn_res = pd.read_excel(mountains_enriched_file)

print()
print("1. Результати для Столиць (capitals_enriched.xlsx):")
print(f"Кількість рядків: {len(df_cap_res)}, Заповнених значень distance: {df_cap_res['distance'].notna().sum()}/{len(df_cap_res)}")
display(df_cap_res) if "display" in globals() else print(df_cap_res.to_string())

print()
print("2. Результати для Гір (mountains_enriched.xlsx):")
print(f"Кількість рядків: {len(df_mtn_res)}, Заповнених значень height: {df_mtn_res['height'].notna().sum()}/{len(df_mtn_res)}")
display(df_mtn_res) if "display" in globals() else print(df_mtn_res.to_string())

print()
print("Тестові набори успішно оброблено, структура збережена, дані збагачено на 100%!")

=== ФІНАЛЬНА ПЕРЕВІРКА РЕЗУЛЬТАТІВ ===

1. Результати для Столиць (capitals_enriched.xlsx):
Кількість рядків: 10, Заповнених значень distance: 10/10
  Capital_From    Country_From Capital_To Country_To  distance
0         Kyiv         Ukraine      Paris     France    2031.9
1       London  United Kingdom       Rome      Italy    1435.6
2        Paris          France     Berlin    Germany     878.0
3       Berlin         Germany     Vienna    Austria     523.9
4       Warsaw          Poland       Kyiv    Ukraine     687.3
5         Rome           Italy     Madrid      Spain    1366.4
6       Madrid           Spain     Lisbon   Portugal     503.9
7       Vienna         Austria   Budapest    Hungary     214.7
8    Stockholm          Sweden       Oslo     Norway     418.8
9       Athens          Greece   Istanbul     Turkey     561.0

2. Результати для Гір (mountains_enriched.xlsx):
Кількість рядків: 10, Заповнених значень height: 10/10
        Mountain         Country   height
0  Mount Ev